# WorkRB Challenge 2026: getting started

A primer on the skill-recommendation task. What it is, why it's hard, what one possible solution looks like, and why scoring well on the standard metrics doesn't mean what you'd hope.

## 1. What is skill extraction?

Given a sentence (job ad, CV bullet, course description), return recommendations of required/provided skills, drawn from a taxonomy, for example [ESCO](https://esco.ec.europa.eu/) (~13.9K skills).

Three examples:

| sentence | gold skill (human identified as relevant in ESCO) |
|---|---|
| *"the ideal candidate should be able to advise customers on sewing patterns"* | `advise customers on sewing patterns` |
| *"diagnose engine vibration during shakedown runs"* | `troubleshoot` |
| *"led a 6-person React/TypeScript squad through a SOC-2 audit"* | `manage staff`, `web programming`, `cyber security` |

## 2. One way to solve it

We can for example train a model that maps every sentence and every skill into the **same vector space**, with the rule:

> if a sentence is about a skill, their vectors should be close.

Once that's true, "recommend the skills in this sentence" is just "find the closest skill vectors to this sentence vector". The whole task collapses to nearest-neighbour search.

<img src="images/biencoder_pipeline.png" alt="bi-encoder pipeline" width="520"/>

One way to learn that space is **contrastive fine-tuning** of an open-source encoder like [MPNET](https://huggingface.co/sentence-transformers/all-mpnet-base-v2). Show the model a batch of `B` matching (sentence, skill) pairs. For every pair, *pull its sentence and its skill together* in the latent space, and *push everything else apart*. Repeat for hundreds of thousands of batches. After enough of them, skills that mean similar things should land near each other and sentences should land near the skills they're about.

The next four cells walk through one concrete instantiation of it (one dataset, one batching scheme, one encoder, one loss) just to make the moving parts visible.

**2.1 a dataset.** One option is [`TechWolf/Synthetic-ESCO-skill-sentences`](https://huggingface.co/datasets/TechWolf/Synthetic-ESCO-skill-sentences): 138K (sentence, skill) pairs generated by prompting an LLM with each ESCO skill and asking for example sentences that would mention it. Each row is one positive: the sentence and its skill belong together. Other choices exist (real annotations, mixed real-plus-synthetic, smaller curated sets), each with their own trade-offs.

In [1]:
import sys

sys.path[:0] = ["..", "../src"]  # repo root + src/, so participant.* and workrb_challenge.* import

from participant.data import SkillSentenceDataset

dataset = SkillSentenceDataset()
print(f"{len(dataset):,} pairs")
for i in range(3):
    sentence, skill = dataset[i]
    print(f"\n[{i}] skill: {skill!r}\n    sent : {sentence!r}")

/Users/warre/Documents/workrb-recsys-hr-challenge-2026/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


138,260 pairs

[0] skill: 'advise customers on sewing patterns'
    sent : 'the ideal candidate for this position should be able to advise customers on sewing patterns based on their needs'

[1] skill: 'advise customers on sewing patterns'
    sent : 'we need an employee who is able to assist our customers with their sewing patterns requests while providing excellent customer service experience.'

[2] skill: 'advise customers on sewing patterns'
    sent : 'if you possess good communication skills and have knowledge of sewing patterns, you can strive as a successful employee in this job'


**2.2 a batching scheme.** The simplest option: sample `B` random positive pairs into a batch. Negatives come for free, every off-diagonal `(sentence_i, skill_j)` for `i ≠ j` is a sentence and skill that don't go together, so the loss can use them as negatives without any extra labelling work. There are smarter batching schemes (hard-negative mining, curriculum, class-balanced sampling); this one is just the default.

In [2]:
from torch.utils.data import DataLoader

from participant.data import default_collate

loader = DataLoader(dataset, batch_size=4, shuffle=True, collate_fn=default_collate)
batch = next(iter(loader))
sentences, skills = batch.queries, batch.targets
for s, k in zip(sentences, skills):
    print(f"{k!r:<55} <- {s!r}")

'satisfy customers'                                     <- 'We are looking for a customer-centric individual who understands the importance of satisfying customers.'
'develop investigation strategy'                        <- 'We require someone who has excellent analytical skills and can develop investigation strategies that are efficient and effective.'
'handle case evidence'                                  <- 'experience in handling case evidence in a manner that meets regulation standards is a must-have for this position'
'operate band saw'                                      <- 'Ability to operate a band saw is a must-have skill for this position.'


**2.3 an encoder.** A pretrained sentence transformer (here, `paraphrase-mpnet-base-v2`) maps any string to a 768-dimensional vector. For simplicity we run the same encoder on both sides (the sentence and the skill name), so they live in the same space by construction. Token-level scoring, untied encoders, and cross-encoders are all reasonable alternatives that change what the model can express.

In [3]:
import torch

from participant.my_model import MyModel, _cosine_similarity

model = MyModel()
model.eval()

with torch.no_grad():
    sentence_emb = model.encode_query(sentences)  # (B, D)
    skill_emb = model.encode_target(skills)       # (B, D)
    sim = _cosine_similarity(sentence_emb, skill_emb)  # (B, B)

print("sentence_emb:", tuple(sentence_emb.shape))
print("skill_emb   :", tuple(skill_emb.shape))
print("\nsim (B x B), diagonal = positives:")
print(sim.round(decimals=3))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9727.25it/s]

sentence_emb: (4, 768)
skill_emb   : (4, 768)

sim (B x B), diagonal = positives:
tensor([[0.6550, 0.2100, 0.1620, 0.0440],
        [0.1790, 0.6510, 0.3300, 0.0560],
        [0.2540, 0.4160, 0.6110, 0.1450],
        [0.1960, 0.2350, 0.1490, 0.6690]])


**2.4 a loss.** Compute cosine similarity between every sentence and every skill in the batch. The result is a `(B, B)` matrix where the diagonal holds the matching pairs and everything else is a non-match. One choice of loss is symmetric InfoNCE: it nudges the encoder to make the diagonal big and the rest small. Triplet, margin softmax, multi-positive variants are all alternatives; what they have in common is some way of saying "matches close, non-matches far".

In [4]:
from participant.data import Batch
from participant.loss import InfoNCELoss

loss_fn = InfoNCELoss(temperature=0.05)
loss = loss_fn(model, Batch(queries=sentences, targets=skills))
print(f"InfoNCE loss on the random batch: {loss.item():.4f}")

InfoNCE loss on the random batch: 0.0047


## 3. How it gets evaluated

For each query sentence, the model scores every skill in the vocabulary, sorts them by score, and the metric checks how high the gold skills land. Current SOTA typically reports three numbers by default: **MAP**, **MRR**, and **RP@K**. They reward different things.

Same toy ranking in all three panels: the gold skills sit at ranks 1 and 4, everything else is a miss.

<img src="images/metrics_walkthrough.png" alt="MAP, MRR, RP@K on the same toy ranking" width="560"/>

Reading off the figure: MAP rewards getting *all* gold skills high. MRR only cares about the *first* gold hit. RP@K just counts gold hits in the top-K, normalised by the number of gold skills. Different metrics, different tolerances, all three pointed at the same target.

The cell below scores that exact toy ranking against the WorkRB metric implementations, so you can see the figure's numbers come out for real.

In [5]:
import numpy as np
from workrb.metrics.ranking import calculate_ranking_metrics

# 1 query, 10 candidate skills.
# Score so that the sorted order is [0, 5, 7, 3, 9, 1, 2, 4, 6, 8].
scores = np.array([[10, 5, 4, 7, 3, 9, 2, 8, 1, 6]], dtype=float)
gold = [[0, 3]]  # skills 0 and 3 are the relevant ones

metrics = calculate_ranking_metrics(
    prediction_matrix=scores,
    pos_label_idxs=gold,
    metrics=["map", "mrr", "rp@10", "rp@2", "recall@5"],
)
for name, value in metrics.items():
    print(f"{name:>10}: {value:.3f}")

       map: 0.750
       mrr: 1.000
     rp@10: 1.000
      rp@2: 0.500
  recall@5: 1.000


All three metrics reward putting gold near the top of the ranking. None of them ask *why* the gold ended up there. The evaluation only cares about the binary assessment of whether a recommended skill is right or wrong, and at what position that skill is recommended. Section 4 looks at why getting that right is easier than it should be.

## 4. The training signal is solvable by surface overlap

The synthetic training data has a leak: the LLM that generated each sentence was prompted with a skill label, so the sentence often contains tokens *from* that label. Why does that matter? Because of how contrastive training actually works.

Recall the setup from section 2.4. We sample a batch of `B` (sentence, skill) positive pairs. The model encodes both sides, computes a `(B, B)` cosine-similarity matrix, and the loss tells it: *the diagonal should be the largest entry in each row*. That's the entire training signal.

If a model can put the diagonal at the top of each row using nothing but **TF-IDF on the raw strings**, the contrastive task is already largely solvable by surface overlap. No semantics needed. Whatever the trained encoder *does* learn beyond TF-IDF, it learns on top of a foundation that can already be cleared by lexical cues.

So let's measure it: sample a real training batch, build the TF-IDF (B, B) similarity matrix that the bi-encoder loss would see, and check how often the diagonal already wins.

In [6]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine
from participant.data import SkillSentenceDataset

# Sample one training batch: B (sentence, skill) positive pairs.
# This mirrors what the InfoNCE loss in section 2.4 sees per step.
B = 64
dataset = SkillSentenceDataset()
rng = np.random.default_rng(0)
rows = rng.choice(len(dataset), size=B, replace=False)
queries     = [dataset[int(i)][0] for i in rows]
skill_pool  = [dataset[int(i)][1] for i in rows]   # the B skills in this batch

# Gold for row i is the diagonal: the i-th skill matches the i-th sentence.
gold_idx = [[i] for i in range(B)]

# Score the (B, B) TF-IDF cosine matrix. No training, no learned parameters.
vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2))
vectorizer.fit(queries + skill_pool)
Q = vectorizer.transform(queries)
S = vectorizer.transform(skill_pool)
scores_tfidf = sk_cosine(Q, S)
print(f"batch size: {B}")
print(f"score matrix: {scores_tfidf.shape}")


batch size: 64
score matrix: (64, 64)


In [7]:
from workrb.metrics.ranking import calculate_ranking_metrics

tfidf_metrics = calculate_ranking_metrics(
    prediction_matrix=scores_tfidf,
    pos_label_idxs=gold_idx,
    metrics=["map", "mrr", "rp@10"],
)
print("TF-IDF (no training, just lexical overlap):")
for name, value in tfidf_metrics.items():
    print(f"  {name:>6}: {value:.3f}")

TF-IDF (no training, just lexical overlap):
     map: 0.858
     mrr: 0.858
   rp@10: 0.891


That number is what the contrastive loss is actually fighting against. Pure surface overlap already places the matching skill at rank 1 most of the time, so a lot of what the encoder "learns" is just confirming patterns TF-IDF already gets right.

The danger isn't that the model picks up lexical cues. It's that the cues are so reliable on this data that the model never has to learn anything *else*. When the test sentence is paraphrased or implicit, the lexical signal disappears, and so does the model's accuracy.

## 5. The lexical trap

Why does TF-IDF score that well? Because the metrics reward putting the gold *anywhere* near the top, and lexical overlap puts it there often enough. The same shortcut bites trained models too: contrastive training on lexically-leaky data teaches the encoder that *shared words* is a good predictor of *belongs together*. Most of the time it is. When it isn't, the model fails in a particular way:

<img src="images/lexical_shortcut.png" alt="the lexical trap" width="600"/>

That kind of failure is invisible to MAP, MRR, and RP@K. The wrong answer beats or is close to the right one because the wrong answer happens to share a token with the sentence. We need an evaluation that asks *whether the recommendations make sense semantically*, not *whether some annotated relevant skill ranks high in a list*.

## 6. Graded evaluation: closing the loophole

One way to close the loophole is to grade predictions on meaning rather than rank. Instead of only yes/no annotation, the labels are ranging from Nonsense all the way to full relevance. This is exactly what this challenge provides.

In [8]:
import json
import textwrap

example = json.load(open("../data/get_started_example.json"))
skills_by_label = {s["label"]: s for s in example["candidate_skills"]}

# the skills that fight for the top of the ranking on the trap query: the gold
# software skill and the two car-bodywork skills the model gets fooled by.
trap_skills = ["debug software", "remove rust from motor vehicles", "operate rust proofing spray gun"]

print("ESCO definitions of the skills involved in the trap query:\n")
for lbl in trap_skills:
    desc = skills_by_label[lbl].get("description", "(no description in data file)")
    print(f"  {lbl}:")
    print(textwrap.fill(desc, width=80, initial_indent="    ", subsequent_indent="    "))
    print()


ESCO definitions of the skills involved in the trap query:

  debug software:
    Repair computer code by analysing testing results, locating the defects
    causing the software to output an incorrect or unexpected result and
    removing these faults.

  remove rust from motor vehicles:
    Remove corrosion from surfaces of motor vehicle bodies.

  operate rust proofing spray gun:
    Operate a semi-automatic or handheld spray gun designed to provide the
    surface of a workpiece with a permanent, corrosion-protective finishing
    coat, safely and according to regulations.



In [9]:
import json
import numpy as np
from workrb.models import ConTeXTMatchModel, CurriculumMatchModel
from workrb.metrics.ranking import calculate_ranking_metrics
from workrb.types import ModelInputType

# 20 real-ESCO candidate skills and 2 queries, picked so classic metrics look
# fine but the rank-1 prediction on the trap query is the wrong skill.
example = json.load(open("../data/get_started_example.json"))
skills    = [s["label"] for s in example["candidate_skills"]]
queries   = example["queries"]
sentences = [q["sentence"] for q in queries]
skill_to_idx = {s: i for i, s in enumerate(skills)}
gold_idx = [[skill_to_idx[g] for g in q["gold_skills"]] for q in queries]

models = {
    "ConTeXT-Match":  ConTeXTMatchModel(),
    "CurriculumMatch": CurriculumMatchModel(),
}

# A tiny mocked judge over (query_id, predicted_skill).
# A real cascade walker would inspect the pair and return 0-4; we hardcode the
# verdicts here so this cell stays self-contained. Default = unsure (label 2).
def mock_cascade_judge(query_id: str, predicted_skill: str) -> int:
    table = {
        ("happy", "advise customers on sewing patterns"): 4,  # human correct
        ("trap",  "debug software"):                       4,  # the actual gold
        ("trap",  "remove rust from motor vehicles"):      0,  # nonsense (cars, not code)
        ("trap",  "operate rust proofing spray gun"):      0,  # nonsense (cars, not code)
        ("trap",  "troubleshoot"):                         3,  # LLM-correct, broader
    }
    return table.get((query_id, predicted_skill), 2)


for name, m in models.items():
    scores = np.asarray(m._compute_rankings(
        queries=sentences, targets=skills,
        query_input_type=ModelInputType.SKILL_SENTENCE,
        target_input_type=ModelInputType.SKILL_NAME,
    ))
    classic = calculate_ranking_metrics(
        prediction_matrix=scores,
        pos_label_idxs=gold_idx,
        metrics=["map", "mrr", "rp@10"],
    )

    print(f"=== {name} ===")
    print(f"  classic  ", end="")
    print("  ".join(f"{k}={v:.2f}" for k, v in classic.items()))

    print(f"  top-3 per query, with mock judge label on rank-1:")
    for qi, q in enumerate(queries):
        order = np.argsort(-scores[qi])
        top1_label = skills[int(order[0])]
        judge = mock_cascade_judge(q["id"], top1_label)
        flag = "  <-- shortcut!" if judge <= 1 else ""
        print(f"    {q['id']:>5}: rank 1 = {top1_label!r}  (sim={float(scores[qi, int(order[0])]):+.3f})  judge={judge}{flag}")
        for r in (1, 2):
            sk = skills[int(order[r])]
            sc = float(scores[qi, int(order[r])])
            star = "  (gold)" if sk in q["gold_skills"] else ""
            print(f"           rank {r+1} = {sk!r}  (sim={sc:+.3f}){star}")
    print()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7702.72it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9202.29it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 26.18it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 25.66it/s]

=== ConTeXT-Match ===
  classic  map=0.75  mrr=0.75  rp@10=1.00
  top-3 per query, with mock judge label on rank-1:
    happy: rank 1 = 'advise customers on sewing patterns'  (sim=+0.877)  judge=4
           rank 2 = 'operate sewing machines'  (sim=+0.442)
           rank 3 = 'Java (computer programming)'  (sim=+0.104)
     trap: rank 1 = 'remove rust from motor vehicles'  (sim=+0.424)  judge=0  <-- shortcut!
           rank 2 = 'debug software'  (sim=+0.404)  (gold)
           rank 3 = 'operate rust proofing spray gun'  (sim=+0.337)

=== CurriculumMatch ===
  classic  map=0.67  mrr=0.67  rp@10=1.00
  top-3 per query, with mock judge label on rank-1:
    happy: rank 1 = 'advise customers on sewing patterns'  (sim=+0.949)  judge=4
           rank 2 = 'operate sewing machines'  (sim=+0.567)
           rank 3 = 'troubleshoot'  (sim=+0.078)
     trap: rank 1 = 'remove rust from motor vehicles'  (sim=+0.524)  judge=0  <-- shortcut!
           rank 2 = 'operate rust proofing spray gun'  (sim

Reading the output: both encoders post respectable classic numbers (RP@10 = 1.00 for both, MAP around 0.7), because for both queries the gold skill lands inside the top-K. But on the trap query the rank-1 prediction is `remove rust from motor vehicles` for both models, the car-bodywork skill we just printed the definition for. CurriculumMatch goes further and puts a *second* car-bodywork skill at rank 2. A meaning-aware judge labels the rank-1 prediction 0 (nonsense), because nothing in the sentence is about removing corrosion from cars.

The similarity numbers tell the same story even more sharply: ConTeXT-Match ranks the trap above the gold by a hair (0.42 vs 0.40), but CurriculumMatch ranks it above the gold by 0.25 cosine points. Both models' embeddings are reading the surface token 'rust' as a stronger signal than the underlying meaning of the whole sentence. That's the lexical shortcut from section 5, made concrete on real ESCO labels.

Scaled up to the full ESCO vocabulary on real job ads, this is the gap between a public-leaderboard MAP and what a downstream user actually gets.

### piling on context doesn't save it

A natural first reaction to that result is *the trap sentence is too short, the model just doesn't have enough to go on*. So let's try a heavily-software-flavored rewrite and see what happens.

In [10]:
import numpy as np
from workrb.models import CurriculumMatchModel
from workrb.types import ModelInputType

pool = [s["label"] for s in example["candidate_skills"]]
variants = {
    "original":           "the engineer will refactor and debug the Rust services this quarter",
    "stuffed-with-context": "the senior backend engineer will refactor and debug the production Rust microservices this quarter",
}

m = CurriculumMatchModel()
scores = np.asarray(m._compute_rankings(
    queries=list(variants.values()), targets=pool,
    query_input_type=ModelInputType.SKILL_SENTENCE,
    target_input_type=ModelInputType.SKILL_NAME,
))

for qi, (vid, sent) in enumerate(variants.items()):
    order = np.argsort(-scores[qi])
    print(f"[{vid}] {sent!r}")
    for r in range(3):
        sk = pool[int(order[r])]
        sc = float(scores[qi, int(order[r])])
        tag = "  (gold)" if sk == "debug software" else ""
        print(f"  r{r+1}: {sk:<40} sim={sc:+.3f}{tag}")
    print()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17229.51it/s]

[original] 'the engineer will refactor and debug the Rust services this quarter'
  r1: remove rust from motor vehicles          sim=+0.524
  r2: operate rust proofing spray gun          sim=+0.419
  r3: debug software                           sim=+0.271  (gold)

[stuffed-with-context] 'the senior backend engineer will refactor and debug the production Rust microservices this quarter'
  r1: remove rust from motor vehicles          sim=+0.392
  r2: operate rust proofing spray gun          sim=+0.325
  r3: debug software                           sim=+0.301  (gold)



Even with *backend*, *production*, *microservices*, the model still ranks `remove rust from motor vehicles` first. The unfamiliar token `Rust` anchors so hard to its training-data sense (corrosion) that the surrounding software vocabulary can't pull the embedding back. Whatever sense the model has of *what those other words mean*, it isn't enough to override the lexical pull of a single out-of-distribution surface form.